 # Submitting a SALT observation request

 This example illustrates how to submit an observation request to the Southern African Large Telescope (SALT). It assumes that you already had a look at the notebook for LCO anmd ESO requests.

## Configuration

You need four pieces of configuration details:

1. Your SALT username. This is defined as the `salt_username` property of `aeonlib.conf.Settings`.
2. Your SALT password. This is defined as the `salt_password` property of `aeonlib.conf.Settings`.
3. The proposal code. This must be the proposal code of an existing proposal, and you must be allowed to resubmit this proposal.
4. The semester in the format yyyy-s, with the year yyyy and the semester (1 or 2) s.

In [1]:
from datetime import datetime, timedelta
from pprint import pprint
from time import sleep

from astropy import units as u
from pyastrosalt.submission import SubmissionStatus

from aeonlib.models import Window
from aeonlib.salt.facility import SALTFacility
from aeonlib.salt.models import SaltSiderealTarget, MagnitudeRange, Hrs, HrsDetector, Constraints, Acquisition, Block, \
    Request

# Replace with the correct proposal code and semester
proposal_code = "2025-2-DDT-005"
semester = "2026-1"

## Create the observation request

The observation request is defined using models from `aeonlib.salt.models`. This example is for illustrative purposes only.

In [2]:
# Observe the Sombrero Galaxy
sombrero = SaltSiderealTarget(
    name="Sombrero Galaxy",
    type="ICRS",
    ra="12h 39m 59.4314s",
    dec="−11° 37′ 23.118",
    target_type="Galaxy",
    magnitude_range=MagnitudeRange(min_magnitude=8.0, max_magnitude=8.0, bandpass="V")
)

# Use the High Resolution Spectrograph
blue_arm = HrsDetector(exposure_times=[10 * u.s])
red_arm = HrsDetector(exposure_times=[10 * u.s])
hrs = Hrs(mode="low resolution", blue_arm=blue_arm, red_arm=red_arm)

# Define observing constraints
constraints = Constraints(
    transparency="clear",
    max_lunar_phase_percentage=50,
    min_lunar_distance=30 * u.deg,
    max_seeing=2,
)

# Define the acquisition. Finder charts are optional
acquisition = Acquisition(
    finder_charts=[],
    position_angle=50 * u.deg,
)

# Window bounds can be specified as datetimes or Astropy.time.Time objects
window = Window(start=datetime.now() + timedelta(days=1), end=datetime.now() + timedelta(days=30))

# Define the observation block
block = Block(
    name=f"Sombrero Galaxy",
    priority=1,
    ranking="high",
    num_visits=1,
    constraints=constraints,
    windows=[window],
    target=sombrero,
    acquisition=acquisition,
    instrument=hrs,
)

request = Request(proposal_code=proposal_code, semester="2025-2", blocks=[block])

pprint(request.model_dump())


{'blocks': [{'acquisition': {'do_not_flip_position_angle': False,
                             'exposure_time': np.float64(1.0),
                             'filter': 'Johnson V',
                             'finder_charts': [],
                             'include_focused_image': False,
                             'position_angle': np.float64(50.0),
                             'reference_star': None},
             'comments': None,
             'constraints': {'max_lunar_phase_percentage': 50.0,
                             'max_seeing': 2.0,
                             'min_lunar_distance': np.float64(30.0),
                             'transparency': 'Clear'},
             'data_notification': 'Normal',
             'identifier': '45ca2cd5-9942-4120-a2d9-ff686e9b0a30',
             'instrument': {'blue_arm': {'exposure_times': [np.float64(10.0)]},
                            'fibre_separation': np.float64(0.016666666666666666),
                            'instrument_name': '

## Validate the observation request

The observation request can be validated with the `validate` method of the SALT facility.

Note that we call the facility constructor with its `use_playground` parameter set to `True`. This ensures that a test server rather than the production server is used. The default is to use the production server.

As your observation request is sent to the server, it may take a few moments before the validation completes.

In [3]:
facility = SALTFacility(use_playground=True)

valid, errors = facility.validate(request)

if valid:
    print("The observation request is valid.")
else:
    print("Validation of the observation request failed with the following error(s):")
    for error in errors:
        print(error)

environ({'COMMAND_MODE': 'unix2003', 'TERM_SESSION_ID': 'ec268349-b3f9-4a8d-b0e0-bc976fb4745d', 'SHELL': '/bin/zsh', 'TMPDIR': '/var/folders/3l/v9_k6g5s1174c0nz5dz00jv80000gn/T/', '__CFBundleIdentifier': 'com.jetbrains.pycharm', 'ENABLE_IDE_INTEGRATION': 'true', 'JEDITERM_SOURCE_ARGS': '', 'HOME': '/Users/christian', 'PATH': '/Users/christian/IdeaProjects/AEONlib/.venv/bin:/Users/christian/.bun/bin:/Users/christian/.pyenv/shims:/Users/christian/Documents/Scripts:/Library/PostgreSQL/13/bin:/usr/local/Cellar/bash/5.2.21/bin:/usr/local/opt/mysql-client@5.7/bin:/usr/local/opt/llvm/bin:/usr/local/Cellar/postgresql@18/18.3/bin:/Library/Frameworks/Python.framework/Versions/3.13/bin:/Library/Frameworks/Python.framework/Versions/3.11/bin:/Library/Frameworks/Python.framework/Versions/2.7/bin:/Library/Frameworks/Python.framework/Versions/3.7/bin:/Library/Frameworks/Python.framework/Versions/3.9/bin:/Library/Frameworks/Python.framework/Versions/3.8/bin:/Users/christian/.deno/bin:/Users/christian/.

ValueError: salt_username is not set.

## Submit the observation request

The observation request can be submitted with the `submit` method of the SALT facility.

This method returns a `pyastrosalt.submission.Submission` object, which you can use to track the submission progress.

In [7]:
submission = facility.submit(request)

shown_log_messages = 0
while submission.status == SubmissionStatus.IN_PROGRESS:
    sleep(1)
    log_entries = submission.log
    for entry in log_entries[shown_log_messages:]:
        print(f"[{str(entry.message_type)}] {entry.message}")
    shown_log_messages = len(log_entries)

if submission.status == SubmissionStatus.SUCCESS:
    print(
        f"The submission was successful. The proposal code is {submission.proposal_code}."
    )
else:
    print("The submission failed with the following error message:")
    print(submission.error)

NameError: name 'facility' is not defined